In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../"))

In [ ]:
def read_csv_file(file_path):
    """
    Reads a CSV file from the given path and returns a pandas DataFrame.

    Parameters:
    file_path (str): Path to the CSV file.

    Returns:
    pd.DataFrame: DataFrame containing the CSV data.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


classifier_metrics = read_csv_file("../../data/evaluation/classifier-basic.csv")

In [ ]:
cla1 = classifier_metrics[classifier_metrics["classifier_architecture"] == "conv_v1"]
cla2 = classifier_metrics[classifier_metrics["classifier_architecture"] == "conv_v2"]
cla3 = classifier_metrics[classifier_metrics["classifier_architecture"] == "conv_v3"]
cla4 = classifier_metrics[classifier_metrics["classifier_architecture"] == "conv_v4"]


In [ ]:
# ── Daten ──────────────────────────────────────────────────────────────────────
architectures = [
    r"$\mathrm{Cla}_1$",
    r"$\mathrm{Cla}_2$",
    r"$\mathrm{Cla}_3$",
    r"$\mathrm{Cla}_4$",
]

ctx_sizes = ["0", "16", "32", "64"]

classes = ["Ball", "Elfmeterpunkt", "Linienkreuzung"]


# AP-Werte [Architektur × Klasse]  →  hier Beispielwerte, einfach ersetzen
ap_values = {
    "Ball": [
        float(cla1["ball_ap"].iloc[0]),
        float(cla2["ball_ap"].iloc[0]),
        float(cla3["ball_ap"].iloc[0]),
        float(cla4["ball_ap"].iloc[0]),
    ],
    "Elfmeterpunkt": [
        float(cla1["penaltyMark_ap"].iloc[0]),
        float(cla2["penaltyMark_ap"].iloc[0]),
        float(cla3["penaltyMark_ap"].iloc[0]),
        float(cla4["penaltyMark_ap"].iloc[0]),
    ],
    "Linienkreuzung": [
        float(cla1["intersections_ap"].iloc[0]),
        float(cla2["intersections_ap"].iloc[0]),
        float(cla3["intersections_ap"].iloc[0]),
        float(cla4["intersections_ap"].iloc[0]),
    ],
}

euc_values = {
    "Ball": [
        float(cla1["classifier_euc_error_ball"].iloc[0]),
        float(cla2["classifier_euc_error_ball"].iloc[0]),
        float(cla3["classifier_euc_error_ball"].iloc[0]),
        float(cla4["classifier_euc_error_ball"].iloc[0]),
    ],
    "Elfmeterpunkt": [
        float(cla1["classifier_euc_error_penaltyMark"].iloc[0]),
        float(cla2["classifier_euc_error_penaltyMark"].iloc[0]),
        float(cla3["classifier_euc_error_penaltyMark"].iloc[0]),
        float(cla4["classifier_euc_error_penaltyMark"].iloc[0]),
    ],
    "Linienkreuzung": [
        float(cla1["classifier_euc_error_intersections"].iloc[0]),
        float(cla2["classifier_euc_error_intersections"].iloc[0]),
        float(cla3["classifier_euc_error_intersections"].iloc[0]),
        float(cla4["classifier_euc_error_intersections"].iloc[0]),
    ],
}

# Mittelwerte
map_values = [np.mean([ap_values[cls][i] for cls in classes]) for i in range(len(architectures))]
mean_euc = [np.mean([euc_values[cls][i] for cls in classes]) for i in range(len(architectures))]


# ── Gemeinsame Layoutparameter ─────────────────────────────────────────────────
x = np.arange(len(architectures))
n_classes = len(classes)
bar_width = 0.22
offsets = np.linspace(-(n_classes - 1) / 2, (n_classes - 1) / 2, n_classes) * bar_width
colors = ["#0072B2", "#E69F00", "#009E73"]  # Okabe-Ito
mean_color = "#CC79A7"


# ── Hilfsfunktion: Achsen stylen ───────────────────────────────────────────────
def style_ax(ax, top_labels=None, top_xlabel=None):
    ax.set_xticks(x)
    ax.set_xticklabels(ctx_sizes, fontsize=11)
    ax.set_xlabel(r"$n_{\mathrm{ctx}}$", fontsize=11)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    if top_labels is not None:
        ax_top = ax.twiny()
        ax_top.set_xlim(ax.get_xlim())  # gleicher Bereich wie untere Achse
        ax_top.set_xticks(x)
        ax_top.set_xticklabels(top_labels, fontsize=11)
        ax_top.spines[["top", "right"]].set_visible(True)
        ax_top.spines[["bottom", "left"]].set_visible(False)
        if top_xlabel:
            ax_top.set_xlabel(top_xlabel, fontsize=11)


# ── Hilfsfunktion: Balken zeichnen ─────────────────────────────────────────────
def draw_bars(ax, data_dict, fmt=".2f"):
    for cls, color, offset in zip(classes, colors, offsets, strict=True):
        bars = ax.bar(
            x + offset,
            data_dict[cls],
            width=bar_width,
            label=cls,
            color=color,
            edgecolor="white",
            linewidth=0.6,
            zorder=3,
        )
        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                f"{bar.get_height():{fmt}}",
                ha="center",
                va="bottom",
                fontsize=10,
                color="#333333",
            )


# ── Hilfsfunktion: Mittelwert-Segmente ────────────────────────────────────────
def draw_mean_segments(ax, mean_vals, label):
    for xi, mv in zip(x, mean_vals, strict=True):
        ax.hlines(
            mv,
            xi - 0.35,
            xi + 0.35,
            colors=mean_color,
            linewidth=2.5,
            linestyle="--",
            zorder=4,
            label=label if xi == 0 else "",
        )
        ax.text(xi + 0.37, mv, f"{mv:.2f}", ha="left", va="center", fontsize=10, color=mean_color)


# — Plot 1: AP / mAP ————————————————————————————————————————————————————————————
fig, ax = plt.subplots(figsize=(9, 5))
draw_bars(ax, ap_values, fmt=".2f")
ax.set_ylim(0, 1.15)
draw_mean_segments(ax, map_values, "Ø Gesamt")
style_ax(ax, top_labels=architectures, top_xlabel="Architektur")
ax.set_ylabel("AP / mAP", fontsize=11)
# ax.set_title("Average Precision (AP)", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.savefig("../../plots/n_ctx_arch/cla_ctx_ap_plot.pdf")

# — Plot 2: Euklidischer Fehler ————————————————————————————————————————————————
fig, ax = plt.subplots(figsize=(9, 5))
ymax = max(v for vals in euc_values.values() for v in vals) * 1.2
draw_bars(ax, euc_values, fmt=".1f")
ax.set_ylim(0, ymax)
draw_mean_segments(ax, mean_euc, "Ø Gesamt")
style_ax(ax, top_labels=architectures, top_xlabel="Architektur")
ax.set_ylabel(r"$\bar{e}$ [px]", fontsize=11)
# ax.set_title("Durchschnittlicher Euklidischer Fehler", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.savefig("../../plots/n_ctx_arch/cla_ctx_euc_plot.pdf")

plt.tight_layout()
plt.show()